In [38]:
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv 
import os

load_dotenv("../rock/.env")                       #Cargamos nuestras contraseñas en el .env
SQL_pass = os.getenv("SQL")

In [39]:
def conectar_db():
    # Ajusta tus credenciales
    usuario = 'root'
    password = SQL_pass
    host = "127.0.0.1"
    db = 'music_db' # El nombre que pusiste en el CREATE SCHEMA
    
    engine = create_engine(f'mysql+mysqlconnector://{usuario}:{password}@{host}/{db}')
    return engine

In [40]:
def subir_artistas(path_csv1, engine):
    df = pd.read_csv(path_csv1)
    
    # 1. Renombrar y limpiar (como antes)
    df = df.rename(columns={
        'id_artist': 'id_artist',
        'artist': 'artist_name',
        'bio': 'artist_bio',
        'listeners': 'artist_listeners',
        'playcount': 'artist_playcount',
        'similar_artist': 'similar_artist'
    })
    df['artist_name'] = df['artist_name'].str.strip()
    
    # 2. ELIMINAR DUPLICADOS DENTRO DEL PROPIO CSV
    df = df.drop_duplicates(subset=['id_artist'])
    
    # 3. COMPROBAR QUÉ ARTISTAS YA ESTÁN EN SQL (Para no repetir)
    try:
        artistas_en_db = pd.read_sql('SELECT id_artist FROM artists', con=engine)
        # Filtramos el dataframe: solo nos quedamos con los que NO están en la DB
        df = df[~df['id_artist'].isin(artistas_en_db['id_artist'])]
    except:
        # Si la tabla está vacía o no existe, seguimos con todo el DF
        pass

    if not df.empty:
        df.to_sql('artists', con=engine, if_exists='append', index=False)
        print(f"✅ Se han subido {len(df)} artistas nuevos.")
    else:
        print("ℹ️ No hay artistas nuevos para subir.")
    
    # Devolvemos todos los artistas que hay ahora en la DB para el cruce de tracks
    return pd.read_sql('SELECT * FROM artists', con=engine)

In [41]:

def procesar_y_subir_tracks(lista_csv_generos, engine):
    # 1. Traemos los artistas directamente de la base de datos para estar seguros
    print("Obteniendo IDs de artistas desde SQL...")
    df_artists_db = pd.read_sql('SELECT id_artist, artist_name FROM artists', con=engine)
    
    # 2. Diccionario de géneros (asegúrate de que coincida con tu tabla 'genre')
    mapeo_generos = {
        "rock": 1,
        "indie": 2,
        "latin": 3,
        "reggaeton": 4,
        "hip-hop": 5
    }

    for archivo in lista_csv_generos:
        try:
            print(f"Procesando {archivo}...")
            df_tracks = pd.read_csv(archivo)
            
            # Limpieza y preparación
            df_tracks['artist'] = df_tracks['artist'].str.strip()
            df_tracks['genre'] = df_tracks['genre'].str.strip().str.lower()

            # Cruzar con los artistas de la DB
            df_unido = df_tracks.merge(df_artists_db, 
                                       left_on='artist', 
                                       right_on='artist_name', 
                                       how='left')

            # Mapear el ID del género
            df_unido['id_genre'] = df_unido['genre'].map(mapeo_generos)

            # Seleccionar columnas exactas del esquema SQL
            df_final = df_unido[['id_artist', 'id_genre', 'track_name', 'year']]
            
            # Quitar nulos (artistas o géneros que no coincidan)
            df_final = df_final.dropna(subset=['id_artist', 'id_genre'])

            # --- CONTROL DE DUPLICADOS ---
            try:
                existentes = pd.read_sql('SELECT id_artist, track_name FROM tracks', con=engine)
                # Combinamos ID y Nombre para identificar la canción única
                df_final['check'] = df_final['id_artist'].astype(str) + df_final['track_name']
                existentes['check'] = existentes['id_artist'].astype(str) + existentes['track_name']
                
                df_final = df_final[~df_final['check'].isin(existentes['check'])]
                df_final = df_final.drop(columns=['check'])
            except:
                pass # Si la tabla está vacía, no hay nada que filtrar

            # Subir a SQL
            if not df_final.empty:
                df_final.to_sql('tracks', con=engine, if_exists='append', index=False)
                print(f"✅ {len(df_final)} canciones nuevas subidas desde {archivo}.")
            else:
                print(f"ℹ️ No había canciones nuevas en {archivo}.")

        except Exception as e:
            print(f"❌ Error procesando {archivo}: {e}")

In [42]:
# 1. Conectar
engine = conectar_db()

In [43]:
# 2. Subir Artistas y obtener el DF con IDs
df_maestro_artistas = subir_artistas('resultado_reggaeton.csv', engine)

✅ Se han subido 62 artistas nuevos.


In [44]:
# 2. Subir Artistas y obtener el DF con IDs
df_maestro_artistas = subir_artistas('resultado_latin.csv', engine)

✅ Se han subido 296 artistas nuevos.


In [45]:
# 2. Subir Artistas y obtener el DF con IDs
df_maestro_artistas = subir_artistas('resultado_indie.csv', engine)

✅ Se han subido 379 artistas nuevos.


In [46]:
df_maestro_artistas = subir_artistas('resultado_rock.csv', engine)


✅ Se han subido 495 artistas nuevos.


In [47]:
df_maestro_artistas = subir_artistas('resultado_hiphop.csv', engine)


✅ Se han subido 769 artistas nuevos.


In [48]:
# 3. Lista de tus archivos de géneros
archivos_a_subir = ['lista_reggaeton.csv', 'lista_indie.csv', 'lista_latin.csv', 'lista_rock.csv', 'lista_hiphop.csv']

In [49]:
# Esta función usará el diccionario de géneros y el df_artistas_actualizado
# para convertir nombres en IDs y subir solo lo que no esté repetido.
procesar_y_subir_tracks(archivos_a_subir, engine)

Obteniendo IDs de artistas desde SQL...
Procesando lista_reggaeton.csv...
✅ 150 canciones nuevas subidas desde lista_reggaeton.csv.
Procesando lista_indie.csv...
✅ 987 canciones nuevas subidas desde lista_indie.csv.
Procesando lista_latin.csv...
✅ 1524 canciones nuevas subidas desde lista_latin.csv.
Procesando lista_rock.csv...
✅ 1111 canciones nuevas subidas desde lista_rock.csv.
Procesando lista_hiphop.csv...
✅ 1864 canciones nuevas subidas desde lista_hiphop.csv.
